# Custom Agentic Workflows: Practice Exercise

Build a **Smart Research Assistant** that routes queries to different knowledge sources based on intent detection.

**What you'll implement:**
1. A Wikipedia lookup node (building on Level 2 skills)
2. An updated intent classifier that detects 3 intents
3. A 3-way routing function

**Workflow you'll build:**
```
START -> intent_detection -> [conditional routing] -> web_search (Tavily)  -> END
                                                   -> wikipedia_lookup     -> END
                                                   -> direct_response      -> END
```

**Skills practiced:**
- Extending routing patterns to handle multiple paths
- Integrating Wikipedia tool into a LangGraph workflow
- Building custom agentic workflows with tool selection

In [ ]:
!pip install langchain-core langchain-openai langgraph langchain-tavily wikipedia langchain-community

In [ ]:
import os
from functools import lru_cache

DEFAULT_REQUIRED_KEYS = ("OPENAI_API_KEY", "TAVILY_API_KEY")

@lru_cache(maxsize=1)
def configure_environment(required_keys=None):
    """
    Factory function to configure environment variables.
    Executes once and caches results.
    """
    if required_keys is None:
        required_keys = ("OPENAI_API_KEY", "TAVILY_API_KEY")

    IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ

    if IN_COLAB:
        from google.colab import userdata
        print("Configuring for Google Colab environment...")
        for key in required_keys:
            try:
                os.environ[key] = userdata.get(key)
            except Exception:
                print(f"Warning: Could not find {key} in Colab secrets.")
    else:
        from dotenv import load_dotenv
        print("Configuring for local environment...")
        load_dotenv()

    # Validation
    for key in required_keys:
        if not os.getenv(key):
            raise ValueError(f"Missing required environment variable: {key}")

    return True

## Setup

Run this cell to import all required libraries and configure the environment.

In [ ]:
# Setup - run this cell first

import os
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_tavily import TavilySearch
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# Initialize environment using the factory function
try:
    configure_environment(("OPENAI_API_KEY", "TAVILY_API_KEY"))
    print("Setup complete!")
except Exception as e:
    print(f"Setup failed: {e}")

Configuring for Google Colab environment...
Setup complete!


## Context

You are extending the Smart Query Router from the guided practice to include **Wikipedia** as a knowledge source. The system should intelligently route queries to:

- **Web Search (Tavily)**: Current events, news, real-time information, recent developments
- **Wikipedia**: Historical facts, scientific concepts, biographies, encyclopedic knowledge  
- **Direct LLM**: Coding help, explanations, creative tasks, general assistance

**Examples:**
| Query | Route | Reason |
|-------|-------|--------|
| "Latest news about SpaceX" | web_search | Current events |
| "Who was Marie Curie?" | wikipedia | Historical biography |
| "How do I write a for loop in Python?" | direct | Coding help |
| "What is quantum entanglement?" | wikipedia | Scientific concept |
| "Current stock price of Apple" | web_search | Real-time data |

## Provided Code: State, LLM, and Tools

The state schema, LLM initialization, and Tavily search tool are provided for you.

In [ ]:
# State schema - provided for you

class AgentState(TypedDict):
    """State schema for our research assistant workflow."""
    query: str           # User's original query
    intent: str          # Detected intent: 'search', 'encyclopedia', or 'direct'
    search_results: str  # Results from Tavily or Wikipedia
    response: str        # Final response to user

print("State schema defined!")

State schema defined!


In [ ]:
# LLM and Tavily search - provided for you

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
tavily_search = TavilySearch(max_results=3, topic="general")

print("LLM and Tavily search initialized!")

LLM and Tavily search initialized!


In [ ]:
# Intent classification schema - provided for you

class IntentClassification(BaseModel):
    """Schema for intent classification with 3 possible intents."""

    intent: Literal["search", "encyclopedia", "direct"] = Field(
        description="The detected intent: "
        "'search' for current events, news, real-time information; "
        "'encyclopedia' for historical facts, scientific concepts, biographies, definitions; "
        "'direct' for coding help, explanations, creative tasks, general assistance"
    )
    reasoning: str = Field(
        description="Brief explanation of why this intent was chosen"
    )

intent_classifier = llm.with_structured_output(IntentClassification)

print("Intent classifier ready!")

Intent classifier ready!


In [ ]:
# Web search node - provided for you

def web_search_node(state: AgentState) -> AgentState:
    """
    Performs web search using Tavily and generates a response.
    """
    query = state["query"]
    print(f"Web Search: Searching for '{query}'")

    # Perform search
    search_response = tavily_search.invoke({"query": query})
    search_results = search_response.get("results", [])

    # Format results
    formatted_results = "\n\n".join([
        f"Title: {r.get('title', 'N/A')}\nURL: {r.get('url', 'N/A')}\nContent: {r.get('content', '')}"
        for r in search_results
    ])

    # Generate response
    messages = [
        SystemMessage(content="You are a helpful assistant. Answer the question using the search results. Cite sources."),
        HumanMessage(content=f"Question: {query}\n\nSearch Results:\n{formatted_results}")
    ]
    response = llm.invoke(messages)

    return {
        **state,
        "search_results": formatted_results,
        "response": response.content
    }

print("Web search node defined!")

Web search node defined!


In [ ]:
# Direct response node - provided for you

def direct_response_node(state: AgentState) -> AgentState:
    """
    Generates a direct response using the LLM's knowledge.
    """
    query = state["query"]
    print(f"Direct Response: Answering '{query}'")

    messages = [
        SystemMessage(content="You are a helpful assistant. Provide clear, accurate answers."),
        HumanMessage(content=query)
    ]
    response = llm.invoke(messages)

    return {**state, "response": response.content}

print("Direct response node defined!")

Direct response node defined!


---

## Part 1: Create the Wikipedia Tool and Node

Create a Wikipedia lookup node that searches Wikipedia and generates a response based on the results.

**Requirements:**
1. Create a `WikipediaAPIWrapper` with `top_k_results=2` and `doc_content_chars_max=2000`
2. Create a `WikipediaQueryRun` tool using the wrapper
3. In the node function:
   - Use the Wikipedia tool to search for the query
   - Generate a response using the LLM with the Wikipedia results
   - Return the updated state

**Hint:** The Wikipedia tool is invoked with `wikipedia_tool.invoke(query)` and returns a string of results.

In [ ]:
# TODO: Create the Wikipedia tool
# Step 1: Create WikipediaAPIWrapper with top_k_results=2, doc_content_chars_max=2000
# Step 2: Create WikipediaQueryRun with the api_wrapper


def wikipedia_node(state: AgentState) -> AgentState:
    """
    Searches Wikipedia and generates a response based on the results.

    Args:
        state: Current state containing the 'query' field

    Returns:
        Updated state with 'search_results' and 'response' fields
    """
    query = state["query"]
    print(f"Wikipedia: Looking up '{query}'")

    # TODO: Implement the Wikipedia lookup node
    wrapper = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=2000)
    # Step 1: Use wikipedia_tool.invoke(query) to get Wikipedia results
    wikipedia_tool = WikipediaQueryRun(api_wrapper=wrapper)
    wikipedia_results = wikipedia_tool.invoke(query)
    # Step 2: Create messages for the LLM with a system prompt and the Wikipedia results
    #         System prompt should instruct the LLM to answer using Wikipedia information
    messages = [
        SystemMessage(content="You are a helpful assistant. Answer the question using the wikipedia results. Cite sources."),
        HumanMessage(content=f"Question: {query}\n\n wikipedia Results:\n{wikipedia_results}")
    ]
    # Step 3: Invoke the LLM to generate a response
    response = llm.invoke(messages)
    # Step 4: Return updated state with search_results and response

    return {
        **state,
        "search_results": wikipedia_results,
        "response": response.content
    }

print("Wikipedia node defined!")

Wikipedia node defined!


---

## Part 2: Implement the Intent Detection Node

Create the intent detection node that classifies queries into one of three categories.

**Requirements:**
- Use the provided `intent_classifier` (structured output model)
- Classify into: `"search"`, `"encyclopedia"`, or `"direct"`
- Include a system prompt that explains the classification criteria

**Classification criteria:**
- **search**: Current events, news, real-time data, recent developments
- **encyclopedia**: Historical facts, scientific concepts, biographies, definitions
- **direct**: Coding help, explanations, creative tasks, general assistance

In [ ]:
def intent_detection_node(state: AgentState) -> AgentState:
    """
    Analyzes the query and classifies it into one of three intents.

    Args:
        state: Current state containing the 'query' field

    Returns:
        Updated state with 'intent' field set to 'search', 'encyclopedia', or 'direct'
    """
    query = state["query"]

    # TODO: Implement intent detection

    # Step 1: Create a system prompt that explains the three categories:
    #         - search: current events, news, real-time information
    #         - encyclopedia: historical facts, scientific concepts, biographies
    #         - direct: coding help, explanations, creative tasks
    system_prompt = """ You are intent classifier to research assistant.
    classify the query into one of the following three categories

    - SEARCH: current events, news, stocks, real-time information, recent developments
    Examples: "Latest news about Agentic AI", "Current stock price of Apple" ,"Current Weather in New Delhi"

    _ ENCYCLOPEDIA: historical facts, scientific concepts, biographies, definitions
    Examples: "Who was Alan Turing and what were his contributions to computer science?", "What is quantum entanglement?", "Can you give me a brief biography of Marie Curie?"

    -DIRECT: coding help, explanations, creative tasks, general assistance
    "Write a Python script to scrape a website.","Explain the concept of 'opportunity cost' using a simple analogy.","Help me draft an email to a client about a project delay."

    Provide your classification with reasoning
    """
    # Step 2: Create messages list with system and user messages
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=query)
    ]
    # Step 3: Use intent_classifier.invoke(messages) to get classification
    result = intent_classifier.invoke(messages)

    # Step 4: Print the detected intent and reasoning for visibility
    print(f"Intent: {result.intent} \n (Reasoning: {result.reasoning})")
    # Step 5: Return updated state with the intent field
    return {
        **state,
        "intent": result.intent
    }

print("Intent detection node defined!")

Intent detection node defined!


---

## Part 3: Implement the Router Function

Create a routing function that returns the appropriate node name based on the detected intent.

**Requirements:**
- Return `"web_search"` if intent is `"search"`
- Return `"wikipedia"` if intent is `"encyclopedia"`
- Return `"direct_response"` if intent is `"direct"`

In [ ]:
def route_by_intent(state: AgentState) -> Literal["web_search", "wikipedia", "direct_response"]:
    """
    Routes to the appropriate node based on detected intent.

    Args:
        state: Current state containing the 'intent' field

    Returns:
        Node name: 'web_search', 'wikipedia', or 'direct_response'
    """
    # TODO: Implement the routing logic

    # Step 1: Get the intent from state
    intent = state["intent"]
    # Step 2: Print the routing decision for visibility

    # Step 3: Return the appropriate node name based on intent
    if intent == "search":
        return "web_search"
    elif intent == "encyclopedia":
        return "wikipedia"
    elif intent == "direct":
        return "direct_response"

    return "direct_response"

---

## Part 4: Build the Workflow

Assemble the complete workflow with all nodes and edges.

**Requirements:**
1. Add 4 nodes: `intent_detection`, `web_search`, `wikipedia`, `direct_response`
2. Add edge from `START` to `intent_detection`
3. Add conditional edges from `intent_detection` using `route_by_intent`
4. Add edges from all three response nodes to `END`

In [ ]:
def build_research_assistant():
    """
    Build and compile the research assistant workflow.

    Returns:
        Compiled workflow ready to invoke
    """
    # Create the state graph
    workflow = StateGraph(AgentState)

    # TODO: Build the workflow

    # Step 1: Add all four nodes
    #         - "intent_detection" using intent_detection_node
    #         - "web_search" using web_search_node
    #         - "wikipedia" using wikipedia_node
    #         - "direct_response" using direct_response_node
    workflow.add_node("intent_detection", intent_detection_node)
    workflow.add_node("web_search", web_search_node)
    workflow.add_node("wikipedia", wikipedia_node)
    workflow.add_node("direct_response", direct_response_node)
    # Step 2: Add edge from START to "intent_detection"
    workflow.add_edge(START, "intent_detection")
    # Step 3: Add conditional edges from "intent_detection"
    #         Use route_by_intent function with path map:
    #         {"web_search": "web_search", "wikipedia": "wikipedia", "direct_response": "direct_response"}
    workflow.add_conditional_edges("intent_detection",
                                   route_by_intent,
                                   {
                                       "web_search": "web_search",
                                       "wikipedia": "wikipedia",
                                       "direct_response": "direct_response"
                                   }
                                  )
    # Step 4: Add edges from all response nodes to END
    workflow.add_edge("web_search", END)
    workflow.add_edge("wikipedia", END)
    workflow.add_edge("direct_response", END)
    # Compile and return
    return workflow.compile()

print("Workflow builder defined!")

Workflow builder defined!


---

## Run Your Implementation

Test your workflow with different types of queries.

In [ ]:
# Build the workflow
research_app = build_research_assistant()
print("Research assistant built successfully!")

Research assistant built successfully!


In [ ]:
# Test 1: Query requiring web search (current events)
print("=" * 70)
print("TEST 1: Current Events Query (should route to web_search)")
print("=" * 70)

result = research_app.invoke({
    "query": "What are the latest developments in AI regulation?",
    "intent": "",
    "search_results": "",
    "response": ""
})

print(f"\nIntent: {result['intent']}")
print(f"\nResponse:\n{result['response'][:500]}...")

TEST 1: Current Events Query (should route to web_search)
Intent: search 
 (Reasoning: The query is asking for current events and recent developments specifically related to AI regulation, which falls under the category of 'search' for real-time information.)
Web Search: Searching for 'What are the latest developments in AI regulation?'

Intent: search

Response:
Recent developments in AI regulation indicate a significant shift towards more structured oversight, particularly in high-risk areas. Here are some key updates:

1. **Transparency and Disclosure**: There is a growing emphasis on adopting clearer transparency and disclosure requirements for AI systems. This is particularly crucial in sectors deemed high-risk, where accountability and human oversight are essential to mitigate potential harms (Credo AI).

2. **Global Regulatory Landscape**: The reg...


In [ ]:
# Test 2: Query requiring Wikipedia (encyclopedic knowledge)
print("=" * 70)
print("TEST 2: Encyclopedia Query (should route to wikipedia)")
print("=" * 70)

result = research_app.invoke({
    "query": "Who was Alan Turing and what were his contributions to computer science?",
    "intent": "",
    "search_results": "",
    "response": ""
})

print(f"\nIntent: {result['intent']}")
print(f"\nResponse:\n{result['response'][:500]}...")

TEST 2: Encyclopedia Query (should route to wikipedia)
Intent: encyclopedia 
 (Reasoning: The query asks for historical facts and contributions related to Alan Turing, which fits the encyclopedia category.)
Wikipedia: Looking up 'Who was Alan Turing and what were his contributions to computer science?'

Intent: encyclopedia

Response:
Alan Turing (23 June 1912 – 7 June 1954) was a pivotal figure in the fields of mathematics, computer science, and cryptanalysis. He is often referred to as the father of theoretical computer science due to his formulation of the concepts of algorithm and computation through the Turing machine, which serves as a foundational model for general-purpose computers.

Turing's contributions were particularly significant during World War II when he worked at Bletchley Park, the British codebreaking cent...


In [ ]:
# Test 3: Query for direct response (coding help)
print("=" * 70)
print("TEST 3: Coding Query (should route to direct_response)")
print("=" * 70)

result = research_app.invoke({
    "query": "How do I reverse a string in Python?",
    "intent": "",
    "search_results": "",
    "response": ""
})

print(f"\nIntent: {result['intent']}")
print(f"\nResponse:\n{result['response']}")

TEST 3: Coding Query (should route to direct_response)
Intent: direct 
 (Reasoning: The query is asking for coding help on how to reverse a string in Python, which falls under direct assistance.)
Direct Response: Answering 'How do I reverse a string in Python?'

Intent: direct

Response:
You can reverse a string in Python using several methods. Here are a few common ways to do it:

1. **Using Slicing**:
   You can use Python's slicing feature to reverse a string. Here's how:

   ```python
   original_string = "Hello, World!"
   reversed_string = original_string[::-1]
   print(reversed_string)  # Output: !dlroW ,olleH
   ```

2. **Using the `reversed()` function**:
   The `reversed()` function returns an iterator that accesses the given string in the reverse order. You can then join the characters back into a string:

   ```python
   original_string = "Hello, World!"
   reversed_string = ''.join(reversed(original_string))
   print(reversed_string)  # Output: !dlroW ,olleH
   ```

3. **U

---

## Expected Output

If your implementation is correct, you should see:

**Test 1 (Current Events):**
- Intent: `search`
- Routes to `web_search` node
- Response includes current information from Tavily

**Test 2 (Encyclopedia):**
- Intent: `encyclopedia`
- Routes to `wikipedia` node
- Response includes information from Wikipedia about Alan Turing

**Test 3 (Coding):**
- Intent: `direct`
- Routes to `direct_response` node
- Response provides Python code examples